# SI_robustness_gender_ratio.ipynb

**Purpose:**
1. Sweep similarity-threshold pairs (general threshold x Chinese-name threshold)
   and compute the resulting female-to-male ratio of self-promoting researchers
   at each pair -> **Fig. S9 input**.
2. Draw a random sample of borderline name-matches (max_score between 0.5 and
   0.9) for manual/GPT-assisted validation -> feeds into
   `SI_robustness_gpt_validation.ipynb`, which produces the **Fig. S8 input**.

**Manuscript:** Gender differences in online visibility of early-career researchers

**Changes from the original notebook** (`9_revision_2025_namematch_ipynb.ipynb`):
- Fixed a `pd.read_pickle(...)` call on a path ending in `.csv` (the same typo
  pattern found and fixed in `1_self_promotion.ipynb`).
- Added a **fixed random seed** to the sampling step. The original had no seed,
  so re-running it would draw a different random sample every time. This
  matters because `SI_robustness_gpt_validation.ipynb`'s manual
  GPT-disagreement overrides are tied to specific row positions in *one*
  particular sample draw -- without a fixed seed, those overrides would
  silently apply to the wrong rows on a re-run.
- Removed leftover/broken cells that referenced undefined variables (visible
  as `NameError`s in the original notebook), and made the
  `author_12_16`/`author_12_16_early_paper_tw` dependency explicit rather than
  relying on notebook execution order (the original notebook's cells were run
  out of sequence -- execution counts were not monotonic).

**Required input files** (outputs of earlier pipeline stages):
- `2_author_12_16_early_paper_tw.pkl` -- all early-career publications with
  tweet info (population, incl. those with zero tweets)
- `2_namematch_score.pkl` -- name-matching scores for every author-tweeter pair

**Outputs:**
- `1_sample_2.csv` -- 100-row validation sample for manual/GPT review
- `2_gender_ratio_threshold.csv` -- Fig. S9 input (`SI_robustness_S8_S9.R` writes it to `tables/fig_s9.csv`)


In [ ]:
import numpy as np
import pandas as pd

RESULT_DIR = "2_result"
RANDOM_SEED = 2026  # fixed so the validation sample is reproducible

pd.set_option("max_colwidth", 100)
pd.set_option("display.max_columns", None)

### 1. Load inputs

In [ ]:
author_12_16_early_paper_tw = pd.read_pickle(
    f"{RESULT_DIR}/2_author_12_16_early_paper_tw.pkl"
)

author_12_16_tw_noNan = pd.read_pickle(
    f"{RESULT_DIR}/2_namematch_score.pkl"
)

### 2. Helper functions

In [ ]:
def tweet_num(row):
    """Count the number of distinct *other people's* tweets mentioning this
    publication, i.e. excluding the author's own self-promoted tweet(s) if
    self_pro == 1."""
    tweets = row["Original_Tweet_list"].copy()

    if row["self_pro"] == 1:
        self_tweet_positions = sorted(
            i for i, tweeter in enumerate(row["Original_Tweeter_list"])
            if tweeter == row["likely_Tweeter_ID"]
        )
        for offset, pos in enumerate(self_tweet_positions):
            try:
                tweets.pop(pos - offset)
            except IndexError:
                author_id = row["author_id"]
                print(f"Warning: could not remove self-tweet for author_id {author_id}")

    try:
        return len(set(tweets))
    except TypeError:
        return 0

In [ ]:
def threshold_test(matched_scores, all_publications, general_threshold, chinese_threshold):
    """Apply a candidate pair of similarity thresholds to the name-matching
    scores and recompute self_pro (self-promotion flag) for every
    publication, merging back onto the full population of early-career
    publications (so publications with no tweets/no match are correctly
    counted as self_pro = 0).

    Parameters
    ----------
    matched_scores : DataFrame
        Name-matching scores (one row per author-tweeter candidate pair),
        e.g. the `2_namematch_score.pkl` data.
    all_publications : DataFrame
        The full population of early-career publications (with or without
        tweets), e.g. `2_author_12_16_early_paper_tw.pkl`.
    general_threshold : float
        Similarity threshold for non-Chinese names.
    chinese_threshold : float
        Similarity threshold for Chinese (Pinyin) names -- typically higher,
        since Pinyin has a limited set of syllable combinations and is more
        prone to false-positive matches at a given similarity score.
    """
    x = matched_scores.copy()

    x["self_pro"] = (x["max_score"] > general_threshold).astype(int)
    x["self_pro"] = x.apply(
        lambda row: 0 if (row["max_score"] < chinese_threshold) and (row["chinese_name"] == 1)
        else row["self_pro"],
        axis=1,
    )

    x = x[[
        "author_id", "doi_lower", "likely_Tweeter_ID", "Original_Tweet_list",
        "Original_Tweeter_list", "max_score", "most_likely_name", "min_tw_time",
        "others_first", "self_text", "self_tw_time", "tw_handle_self_score",
        "tw_name_self_score", "with_tw", "self_pro", "chinese_name",
    ]].copy()

    x["tw_others_num"] = x.apply(tweet_num, axis=1)
    x = x.drop(columns=["Original_Tweet_list", "Original_Tweeter_list"]).drop_duplicates()

    merged = all_publications.drop_duplicates(["author_id", "doi_lower"]).copy()
    merged["file_num"] = merged["file_num"].fillna(0)
    merged = merged.merge(x, on=["author_id", "doi_lower"], how="left")

    for col in ["self_pro", "with_tw", "tw_others_num", "others_first"]:
        merged[col] = merged[col].fillna(0)

    return merged

### 3. Threshold sweep -> Fig. S9 input

In [ ]:
threshold_result = []
for general_t in np.arange(0.5, 1.01, 0.1):
    for chinese_t in np.arange(0.5, 1.01, 0.1):
        test = threshold_test(
            author_12_16_tw_noNan, author_12_16_early_paper_tw,
            round(general_t, 2), round(chinese_t, 2),
        )
        female_n = test[(test["self_pro"] == 1) & (test["gender"] == "female")]["author_id"].nunique()
        male_n = test[(test["self_pro"] == 1) & (test["gender"] == "male")]["author_id"].nunique()
        threshold_result.append({
            "general_threshold": round(general_t, 2),
            "chinese_threshold": round(chinese_t, 2),
            "gender_ratio": female_n / male_n if male_n > 0 else np.nan,
        })

threshold_result = pd.DataFrame(threshold_result)
threshold_result.to_csv(f"{RESULT_DIR}/2_gender_ratio_threshold.csv", index=False)
print(f"Wrote {RESULT_DIR}/2_gender_ratio_threshold.csv ({len(threshold_result)} threshold pairs)")

### 4. Draw the validation sample for manual/GPT review

> **Fix applied:** the original notebook selected only
> `["author_id","gender","most_likely_name","chinese_name"]` into `x` before
> later filtering on `x["max_score"]` and `x["self_pro"]` -- columns that were
> never selected, which would raise a `KeyError`. Given the notebook's
> non-sequential execution order, this only makes sense if the `author_12_16`
> used there was actually the **output of `threshold_test()` at the paper's
> chosen production thresholds** (general=0.6, Chinese=0.8), which does have
> both columns. This makes that dependency explicit instead of implicit.

In [ ]:
PRODUCTION_GENERAL_THRESHOLD = 0.6
PRODUCTION_CHINESE_THRESHOLD = 0.8

production_result = threshold_test(
    author_12_16_tw_noNan, author_12_16_early_paper_tw,
    PRODUCTION_GENERAL_THRESHOLD, PRODUCTION_CHINESE_THRESHOLD,
)

author_names = (
    author_12_16_tw_noNan[[
        "author_id", "last_name", "first_name", "Index_firstname",
        "first_name_revised", "chinese_name",
    ]]
    .drop_duplicates()
)
author_names = author_names[
    author_names["last_name"].notnull() | author_names["first_name"].notnull()
]

scored = production_result[
    production_result["tw_name_self_score"].notnull()
][["author_id", "gender", "most_likely_name", "chinese_name", "max_score", "self_pro"]].merge(
    author_names, on="author_id", how="left", suffixes=("", "_names")
)

# Borderline cases: similarity score in the ambiguous 0.5-0.9 range, where the
# choice of threshold actually matters.
borderline = scored[(scored["max_score"] >= 0.5) & (scored["max_score"] <= 0.9)]

In [ ]:
# Oversample the (rarer) self-promotion==1 cases so the validation set isn't
# dominated by the majority class; fixed seed for reproducibility (see note
# in the top markdown cell).
validation_sample = pd.concat([
    borderline[borderline["self_pro"] == 0].sample(70, random_state=RANDOM_SEED),
    borderline[borderline["self_pro"] == 1].sample(30, random_state=RANDOM_SEED),
])

validation_sample.to_csv(f"{RESULT_DIR}/1_sample_2.csv", index=False)
print(f"Wrote {RESULT_DIR}/1_sample_2.csv ({len(validation_sample)} rows for manual/GPT review)")